In [1]:
import torch
import torch.nn as nn
import math
import random
import numpy as np

# %% [markdown]
# # Copernicus-FM 模型前向流程演示（以 S1 模态为例）
# 
# 本 notebook 展示 `MaskedAutoencoderViT` 类处理 Sentinel-1 图像的过程：
# - 输入：`[B, 2, 224, 224]`
# - Patch embedding（动态超网络生成卷积核）
# - 添加位置编码 + 元数据编码（简化为可学习 token）
# - 随机掩码（mask_ratio=0.75）
# - Transformer 编码器
# - 解码器 + 预测 patch
# - 计算 MAE 损失

# %%
import torch
import torch.nn as nn
import math
import random
import numpy as np

# ------------------------------------------------------------
# 1. 辅助函数与简化模块
# ------------------------------------------------------------

def get_2d_sincos_pos_embed(embed_dim, grid_size, cls_token=False):
    """生成 2D sin-cos 位置编码（简化版）"""
    grid_h = np.arange(grid_size, dtype=np.float32)
    grid_w = np.arange(grid_size, dtype=np.float32)
    grid = np.meshgrid(grid_w, grid_h)
    grid = np.stack(grid, axis=0)          # (2, grid_size, grid_size)
    grid = grid.reshape([2, 1, grid_size, grid_size])
    
    def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
        assert embed_dim % 2 == 0
        omega = np.arange(embed_dim // 2, dtype=np.float32)
        omega /= embed_dim / 2.
        omega = 1. / 10000 ** omega
        pos = pos.reshape(-1)
        out = np.einsum('m,d->md', pos, omega)
        emb_sin = np.sin(out)
        emb_cos = np.cos(out)
        emb = np.concatenate([emb_sin, emb_cos], axis=1)
        return emb
    
    pos_embed = get_1d_sincos_pos_embed_from_grid(embed_dim, grid[0]) + \
                get_1d_sincos_pos_embed_from_grid(embed_dim, grid[1])
    if cls_token:
        pos_embed = np.concatenate([np.zeros([1, embed_dim]), pos_embed], axis=0)
    return pos_embed


def pi_resize_patch_embed(patch_embed, new_size):
    """简化版：不对卷积核做插值，直接返回原值（实际训练中会做 resize）"""
    return patch_embed


class Block(nn.Module):
    """Transformer Block"""
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=True, norm_layer=nn.LayerNorm):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, bias=qkv_bias, batch_first=True)
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, dim),
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x


class Dynamic_MLP_OFA_spectral(nn.Module):
    """光谱模态动态 patch embedding（简化版，但保留动态生成卷积核的核心逻辑）"""
    def __init__(self, wv_planes=128, inter_dim=128, kernel_size=16, embed_dim=768):
        super().__init__()
        self.kernel_size = kernel_size
        self.embed_dim = embed_dim
        self._num_kernel = kernel_size * kernel_size * embed_dim
        
        # 简化的权重生成器：直接使用可学习的参数代替超网络
        self.weight_gen_linear = nn.Linear(wv_planes, self._num_kernel)
        self.bias_gen_linear = nn.Linear(wv_planes, embed_dim)
        self.scaler = 0.01
        
        # 对波长和带宽的编码（简化为线性层）
        self.wave_proj = nn.Linear(2, wv_planes)  # 输入 [C, 2]（波长+带宽）
        
    def forward(self, img_feat, wvs, bandwidths, kernel_size=None):
        """
        img_feat: [B, C, H, W]
        wvs, bandwidths: [C] 每个通道的波长和带宽
        """
        B, C, H, W = img_feat.shape
        ks = kernel_size if kernel_size is not None else self.kernel_size
        
        # 构建波长+带宽特征 [C, 2]
        wave_features = torch.stack([wvs, bandwidths], dim=1)  # [C, 2]
        waves = self.wave_proj(wave_features)                  # [C, wv_planes]
        
        # 动态生成卷积核权重和偏置
        weight_flat = self.weight_gen_linear(waves)            # [C, _num_kernel]
        bias = self.bias_gen_linear(waves).mean(dim=0)         # [embed_dim]
        
        # 重塑为卷积核 [D, C, ks, ks]
        weight = weight_flat.view(C, ks, ks, self.embed_dim)
        weight = weight.permute(3, 0, 1, 2)                    # [D, C, ks, ks]
        weight = weight * self.scaler
        bias = bias * self.scaler
        
        # 执行卷积
        out = torch.nn.functional.conv2d(img_feat, weight, bias=bias, stride=ks, padding=0)
        # out shape: [B, D, H/ks, W/ks]
        B, D, H_out, W_out = out.shape
        tokens = out.flatten(2).transpose(1, 2)                # [B, L, D], L = H_out*W_out
        return tokens, waves


class Dynamic_MLP_Decoder(nn.Module):
    """解码器预测头（简化）"""
    def __init__(self, decoder_embed=512):
        super().__init__()
        self.linear = nn.Linear(decoder_embed, decoder_embed)
    def forward(self, x, waves, kernel_size=None):
        return self.linear(x)


# ------------------------------------------------------------
# 2. MaskedAutoencoderViT 主类（精简版，但完整包含 MAE 流程）
# ------------------------------------------------------------
class MaskedAutoencoderViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3,
                 embed_dim=768, depth=12, num_heads=12,
                 decoder_embed_dim=512, decoder_depth=8, decoder_num_heads=16,
                 mlp_ratio=4., norm_layer=nn.LayerNorm, norm_pix_loss=False):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.num_patches = (img_size // patch_size) ** 2
        self.norm_pix_loss = norm_pix_loss
        
        # 动态 patch embedding
        self.patch_embed = Dynamic_MLP_OFA_spectral(embed_dim=embed_dim, kernel_size=patch_size)
        
        # 位置编码（固定 sin-cos）
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim), requires_grad=False)
        self._init_pos_embed()
        
        # 元数据编码（简化为可学习 token，实际应使用傅里叶编码）
        self.meta_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # Transformer 编码器
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio, qkv_bias=True, norm_layer=norm_layer)
            for _ in range(depth)
        ])
        self.norm = norm_layer(embed_dim)
        
        # 解码器
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim, bias=True)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, decoder_embed_dim), requires_grad=False)
        self._init_decoder_pos_embed()
        
        self.decoder_blocks = nn.ModuleList([
            Block(decoder_embed_dim, decoder_num_heads, mlp_ratio, qkv_bias=True, norm_layer=norm_layer)
            for _ in range(decoder_depth)
        ])
        self.decoder_norm = norm_layer(decoder_embed_dim)
        self.decoder_pred = Dynamic_MLP_Decoder(decoder_embed=decoder_embed_dim)
        
        self._init_weights()
    
    def _init_pos_embed(self):
        pos_embed = get_2d_sincos_pos_embed(self.pos_embed.shape[-1], int(self.num_patches**0.5), cls_token=True)
        self.pos_embed.data.copy_(torch.from_numpy(pos_embed).float().unsqueeze(0))
    
    def _init_decoder_pos_embed(self):
        pos_embed = get_2d_sincos_pos_embed(self.decoder_pos_embed.shape[-1], int(self.num_patches**0.5), cls_token=True)
        self.decoder_pos_embed.data.copy_(torch.from_numpy(pos_embed).float().unsqueeze(0))
    
    def _init_weights(self):
        torch.nn.init.normal_(self.cls_token, std=.02)
        torch.nn.init.normal_(self.mask_token, std=.02)
        torch.nn.init.normal_(self.meta_token, std=0.02)
        self.apply(self._init_weights_module)
    
    def _init_weights_module(self, m):
        if isinstance(m, nn.Linear):
            torch.nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
    
    def patchify(self, imgs, patch_size=None):
        p = patch_size if patch_size is not None else self.patch_size
        B, C, H, W = imgs.shape
        assert H % p == 0 and W % p == 0
        h = H // p
        w = W // p
        x = imgs.reshape(B, C, h, p, w, p)
        x = torch.einsum('bchpwq->bhwpqc', x)
        x = x.reshape(B, h * w, p * p * C)
        return x
    
    def unpatchify(self, x, patch_size=None):
        p = patch_size if patch_size is not None else self.patch_size
        B, L, _ = x.shape
        h = w = int(math.sqrt(L))
        assert h * w == L
        C = x.shape[2] // (p * p)
        x = x.reshape(B, h, w, p, p, C)
        x = torch.einsum('bhwpqc->bchpwq', x)
        imgs = x.reshape(B, C, h * p, w * p)
        return imgs
    
    def random_masking(self, x, mask_ratio):
        N, L, D = x.shape
        len_keep = int(L * (1 - mask_ratio))
        noise = torch.rand(N, L, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))
        mask = torch.ones([N, L], device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)
        return x_masked, mask, ids_restore
    
    def forward_encoder(self, x, mask_ratio, wvs, bandwidths, kernel_size=None):
        # 1. patch embedding
        x, waves = self.patch_embed(x, wvs, bandwidths, kernel_size)  # [B, L, D]
        print(f"  After patch embed: shape {x.shape}")
        
        # 2. 添加位置编码和元数据编码（简化为加可学习 token）
        pos_embed = self.pos_embed[:, 1:, :]  # 去掉 cls 部分
        x = x + pos_embed + self.meta_token   # [B, L, D]
        print(f"  After adding pos/meta embed: {x.shape}")
        
        # 3. 随机掩码
        x, mask, ids_restore = self.random_masking(x, mask_ratio)
        print(f"  After masking: visible tokens {x.shape}")
        
        # 4. 添加 cls token
        cls_token = self.cls_token + self.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)   # [B, 1+len_keep, D]
        print(f"  After adding cls token: {x.shape}")
        
        # 5. Transformer 编码器
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x, mask, ids_restore, waves
    
    def forward_decoder(self, x, ids_restore, waves, kernel_size=None):
        # 1. 映射到解码器维度
        x = self.decoder_embed(x)              # [B, 1+len_keep, decoder_dim]
        print(f"  Decoder embed: {x.shape}")
        
        # 2. 插入 mask tokens
        mask_tokens = self.mask_token.repeat(x.shape[0], ids_restore.shape[1] + 1 - x.shape[1], 1)
        x_ = torch.cat([x[:, 1:, :], mask_tokens], dim=1)  # 去掉 cls
        x_ = torch.gather(x_, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, x.shape[2]))
        x = torch.cat([x[:, :1, :], x_], dim=1)            # 恢复 cls
        print(f"  After mask token insertion: {x.shape}")
        
        # 3. 添加解码器位置编码
        decoder_pos_embed = self.decoder_pos_embed
        x = x + decoder_pos_embed
        print(f"  After decoder pos embed: {x.shape}")
        
        # 4. 解码器 Transformer
        for blk in self.decoder_blocks:
            x = blk(x)
        x = self.decoder_norm(x)
        
        # 5. 预测 patch 像素
        x = self.decoder_pred(x, waves, kernel_size)   # [B, L+1, decoder_dim]
        x = x[:, 1:, :]                                # 去掉 cls token
        print(f"  After prediction (remove cls): {x.shape}")
        return x
    
    def forward_loss(self, imgs, pred, mask, patch_size=None):
        target = self.patchify(imgs, patch_size)
        if self.norm_pix_loss:
            mean = target.mean(dim=-1, keepdim=True)
            var = target.var(dim=-1, keepdim=True)
            target = (target - mean) / (var + 1e-6) ** 0.5
        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)                     # 每个 patch 的 MSE
        loss = (loss * mask).sum() / mask.sum()      # 只计算被 mask 的 patch
        return loss
    
    def forward(self, imgs, wvs, bandwidths, mask_ratio=0.75, kernel_size=None):
        """
        imgs: [B, C, H, W]  例如 S1: [2, 2, 224, 224]
        wvs, bandwidths: [C] 每个通道的波长和带宽
        """
        print("\n=== 开始前向传播 ===")
        print(f"输入图像 shape: {imgs.shape}")
        print(f"mask_ratio: {mask_ratio}")
        
        latent, mask, ids_restore, waves = self.forward_encoder(imgs, mask_ratio, wvs, bandwidths, kernel_size)
        pred = self.forward_decoder(latent, ids_restore, waves, kernel_size)
        loss = self.forward_loss(imgs, pred, mask, kernel_size)
        print(f"最终 MAE loss: {loss.item():.4f}")
        return loss, pred, mask


# ------------------------------------------------------------
# 3. 实例化模型并测试 S1 输入
# ------------------------------------------------------------
if __name__ == "__main__":
    # 设置设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # 创建模型（ViT-Base 配置）
    model = MaskedAutoencoderViT(
        img_size=224,
        patch_size=16,
        in_chans=2,          # S1 通道数
        embed_dim=768,
        depth=12,
        num_heads=12,
        decoder_embed_dim=512,
        decoder_depth=8,
        norm_pix_loss=False
    ).to(device)
    
    # 构造 S1 的假输入
    B = 2
    C = 2   # VV + VH
    H = 224
    W = 224
    dummy_img = torch.randn(B, C, H, W).to(device)
    
    # S1 的波长和带宽（单位：nm，来自论文附录表9）
    # S1 GRD 中心波长约 5.6e7 nm（实际是频率，但这里用占位值）
    wvs = torch.tensor([5.6e7, 5.6e7], device=device).float()
    bandwidths = torch.tensor([1e9, 1e9], device=device).float()
    
    # 前向传播
    with torch.no_grad():
        loss, pred, mask = model(dummy_img, wvs, bandwidths, mask_ratio=0.75, kernel_size=16)
    
    print("\n=== 前向完成 ===")
    print(f"预测 patch 序列形状: {pred.shape}")   # [B, L, p*p*C]
    print(f"掩码矩阵形状: {mask.shape}")         # [B, L]